# GAT Training notebook

In [7]:
## 1. Setup and Configuration
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
from torch import nn, optim
from sklearn.metrics import f1_score, accuracy_score
import matplotlib.pyplot as plt

# Connect to Neo4j
from neo4j import GraphDatabase

In [8]:
class Config:
    neo4j_uri = "bolt://localhost:7687"
    neo4j_user = "neo4j"
    neo4j_password = "111122223333"
    batch_size = 32
    num_epochs = 200
    lr = 0.005
    hidden_dim = 64
    num_heads = 4
    dropout = 0.6
    weight_decay = 0.001
    
config = Config()

NUM_INPUT_FEATURES = 10
NUM_CLASSES = 5

In [ ]:
from typing import List, Tuple
import torch
from neo4j import GraphDatabase
import networkx as nx
from torch_geometric.data import Data

class GraphDataLoader:
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        
    def load_training_data(self, concept_id: str) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Load training data for a specific concept (number) from Neo4j.
        Similar to the PPI dataset loading but adapted for our graph structure.
        
        Args:
            concept_id: The base concept ID (e.g., "1" for all variations of number 1)
            
        Returns:
            Tuple containing:
            - node_features: Tensor of shape [num_nodes, num_features]
            - node_labels: Tensor of shape [num_nodes, num_labels]
            - edge_index: Tensor of shape [2, num_edges]
        """
        with self.driver.session() as session:
            # Get all training samples for the concept
            query = """
            MATCH (n)
            WHERE n.session_id STARTS WITH $concept_id + '_'
            WITH DISTINCT n.session_id AS session_id
            RETURN collect(session_id) AS training_sessions
            """
            result = session.run(query, concept_id=concept_id)
            training_sessions = result.single()["training_sessions"]
            
            # Lists to store the graph components
            node_features_list = []
            node_labels_list = []
            edge_index_list = []
            num_nodes_seen = 0
            
            # Process each training sample
            for session_id in training_sessions:
                # Query to get the graph structure
                graph_query = """
                MATCH (n {session_id: $session_id})
                WHERE n:Point OR n:Vector
                WITH n, labels(n) as node_labels
                OPTIONAL MATCH (n)-[r]-(m {session_id: $session_id})
                RETURN id(n) as node_id, 
                       node_labels,
                       COLLECT(DISTINCT id(m)) as connected_nodes,
                       n.x as x, 
                       n.y as y,
                       n.angle as angle
                """
                graph_result = session.run(graph_query, session_id=session_id)
                
                # Process nodes and edges
                current_nodes = {}
                current_edges = []
                features = []
                labels = []
                
                for record in graph_result:
                    node_id = len(current_nodes)
                    current_nodes[record["node_id"]] = node_id
                    
                    # Create node features vector [x, y, angle, is_point, is_vector]
                    node_feat = [
                        float(record["x"] or 0),
                        float(record["y"] or 0),
                        float(record["angle"] or 0),
                        1.0 if "Point" in record["node_labels"] else 0.0,
                        1.0 if "Vector" in record["node_labels"] else 0.0
                    ]
                    features.append(node_feat)
                    
                    # Create edges
                    for target_id in record["connected_nodes"]:
                        if target_id in current_nodes:
                            current_edges.append([node_id, current_nodes[target_id]])
                            current_edges.append([current_nodes[target_id], node_id])
                
                if features:
                    # Convert to tensors
                    node_features = torch.tensor(features, dtype=torch.float)
                    edge_index = torch.tensor(current_edges, dtype=torch.long).t()
                    
                    # Adjust edge indices
                    if len(edge_index) > 0:
                        edge_index = edge_index + num_nodes_seen
                    
                    # Create labels (multi-hot encoding for node types)
                    node_labels = torch.zeros((len(features), 5), dtype=torch.float)  # Adjust size based on your label count
                    
                    node_features_list.append(node_features)
                    edge_index_list.append(edge_index)
                    node_labels_list.append(node_labels)
                    
                    num_nodes_seen += len(features)
            
            # Merge all graphs into a single graph with multiple components
            node_features = torch.cat(node_features_list, 0)
            node_labels = torch.cat(node_labels_list, 0)
            edge_index = torch.cat(edge_index_list, 1)
            
            return node_features, node_labels, edge_index

In [ ]:
class GraphDataLoader(DataLoader):
    """
    When dealing with batches it's always a good idea to inherit from PyTorch's provided classes (Dataset/DataLoader).

    """
    def __init__(self, node_features_list, node_labels_list, edge_index_list, batch_size=1, shuffle=False):
        graph_dataset = GraphDataset(node_features_list, node_labels_list, edge_index_list)
        # We need to specify a custom collate function, it doesn't work with the default one
        super().__init__(graph_dataset, batch_size, shuffle, collate_fn=graph_collate_fn)

class GraphDataset(Dataset):
    def __init__(self, node_features_list, node_labels_list, edge_index_list):
        self.node_features_list = node_features_list
        self.node_labels_list = node_labels_list
        self.edge_index_list = edge_index_list
        
    def __len__(self):
        return len(self.edge_index_list)
    
    def __getitem__(self, idx):
        return self.node_features_list[idx], self.node_labels_list[idx], self.edge_index_list[idx]
    
def graph_collate_fn(batch):
    """
    The main idea here is to take multiple graphs from PPI as defined by the batch size
    and merge them into a single graph with multiple connected components.

    It's important to adjust the node ids in edge indices such that they form a consecutive range. Otherwise
    the scatter functions in the implementation 3 will fail.

    :param batch: contains a list of edge_index, node_features, node_labels tuples (as provided by the GraphDataset)
    """

    edge_index_list = []
    node_features_list = []
    node_labels_list = []
    num_nodes_seen = 0

    for features_labels_edge_index_tuple in batch:
        # Just collect these into separate lists
        node_features_list.append(features_labels_edge_index_tuple[0])
        node_labels_list.append(features_labels_edge_index_tuple[1])

        edge_index = features_labels_edge_index_tuple[2]  # all of the components are in the [0, N] range
        edge_index_list.append(edge_index + num_nodes_seen)  # very important! translate the range of this component
        num_nodes_seen += len(features_labels_edge_index_tuple[1])  # update the number of nodes we've seen so far

    # Merge the PPI graphs into a single graph with multiple connected components
    node_features = torch.cat(node_features_list, 0)
    node_labels = torch.cat(node_labels_list, 0)
    edge_index = torch.cat(edge_index_list, 1)

    return node_features, node_labels, edge_index